In [4]:
import json

mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"

with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("📝 JSON VERİ YAPISI ÖRNEĞİ:\n")

# Eğer JSON bir liste ise ilk iki elemanı basalım
if isinstance(mapping_data, list):
    print("Veri yapısı bir LİSTE.")
    print(json.dumps(mapping_data[:2], indent=4, ensure_ascii=False))
# Eğer JSON bir sözlük (dictionary) ise ilk iki anahtarı ve içeriğini basalım
elif isinstance(mapping_data, dict):
    print("Veri yapısı bir SÖZLÜK.")
    ilk_iki_anahtar = list(mapping_data.keys())[:2]
    ornek_dict = {k: mapping_data[k] for k in ilk_iki_anahtar}
    print(json.dumps(ornek_dict, indent=4, ensure_ascii=False))

📝 JSON VERİ YAPISI ÖRNEĞİ:

Veri yapısı bir SÖZLÜK.
{
    "00_02_49": {
        "annotations": [
            {
                "class": "Normal",
                "note": "",
                "timestamp": "2025-01-29T23:25:47.373454",
                "ranges": []
            }
        ]
    },
    "00_14_08": {
        "annotations": [
            {
                "class": "Normal",
                "note": "",
                "timestamp": "2025-01-29T23:26:42.929497",
                "ranges": []
            }
        ]
    }
}


In [5]:
import os
import json

# mapping.json dosyanın gerçek yolu
mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"

with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("🔍 VERİ SETİNDEKİ SİBER SALDIRI VE ARIZA TÜRLERİ ANALİZ EDİLİYOR...\n")

saldiri_kategorileri = {}

for ucus_adi, ucus_bilgisi in mapping_data.items():
    # 'annotations' listesini güvenli bir şekilde alıyoruz
    annotations = ucus_bilgisi.get("annotations", [])
    
    if annotations:
        # Listenin ilk elemanının içindeki 'class' değerini (yani etiketini) çekiyoruz
        label = annotations[0].get("class", "Normal")
        
        # Eğer etiket 'Normal' değilse, yani bir saldırı veya arızaysa kategorilere ekle
        if label != "Normal":
            if label not in saldiri_kategorileri:
                saldiri_kategorileri[label] = []
            saldiri_kategorileri[label].append(ucus_adi)

print("==========================================================================================")
print("📊 BULUNAN SALDIRI / ANOMALİ KATEGORİLERİ VE DOSYA SAYILARI 📊")
print("==========================================================================================")

if len(saldiri_kategorileri) == 0:
    print("⚠️ Dikkat: Hâlâ anomalili veri gruplanamadı. JSON içeriğinde 'Normal' dışında bir sınıf olmayabilir.")
else:
    for kat_adi, dosyalar in saldiri_kategorileri.items():
        print(f"🚨 Saldırı/Arıza Türı: {kat_adi}")
        print(f"   ↳ Sistemde toplam {len(dosyalar)} adet bu türde uçuş var.")
        print(f"   ↳ Örnek İlk 2 Dosya İsimleri: {dosyalar[:2]}")
        print("-" * 90)

🔍 VERİ SETİNDEKİ SİBER SALDIRI VE ARIZA TÜRLERİ ANALİZ EDİLİYOR...

📊 BULUNAN SALDIRI / ANOMALİ KATEGORİLERİ VE DOSYA SAYILARI 📊
🚨 Saldırı/Arıza Türı: Uncategorized
   ↳ Sistemde toplam 141 adet bu türde uçuş var.
   ↳ Örnek İlk 2 Dosya İsimleri: ['2018-05-24/19_06_07', '2018-05-24/19_48_57']
------------------------------------------------------------------------------------------
🚨 Saldırı/Arıza Türı: External Position
   ↳ Sistemde toplam 194 adet bu türde uçuş var.
   ↳ Örnek İlk 2 Dosya İsimleri: ['2018-05-24/19_57_23', '2018-05-26/16_19_20']
------------------------------------------------------------------------------------------
🚨 Saldırı/Arıza Türı: Altitude
   ↳ Sistemde toplam 77 adet bu türde uçuş var.
   ↳ Örnek İlk 2 Dosya İsimleri: ['22_00_49', 'log_13_2021-10-3-16-05-24']
------------------------------------------------------------------------------------------
🚨 Saldırı/Arıza Türı: Mechanical
   ↳ Sistemde toplam 43 adet bu türde uçuş var.
   ↳ Örnek İlk 2 Dosya İsimle

In [6]:
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# ==========================================================================================
# 1. AYARLAR VE GEREKLİ YOLLAR
# ==========================================================================================
mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"
# NOT: Saldırılı/Ham dosyalarının işlenmiş hallerinin olduğu ana dizin yolu
processed_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\cleaned_train" # Yolunuzu projenize göre güncelleyin kanka
model_path = r"C:\Users\feyza\Desktop\uav_project\models\uav_autoencoder_v1.pth"

features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

# Model Mimarisi (Kayıttan okumak için iskelet şart)
class UAVFinalAutoencoder(nn.Module):
    def __init__(self, dim):
        super(UAVFinalAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Sabit Modeli Yükleme
model = UAVFinalAutoencoder(dim=30)
model.load_state_dict(torch.load(model_path))
model.eval()

# ==========================================================================================
# 2. MAPPING'DEN GPS SPOOFING (EXTERNAL POSITION) DOSYALARINI AYIKLAMA
# ==========================================================================================
with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

gps_attack_folders = []
for ucus_adi, ucus_bilgisi in mapping_data.items():
    annotations = ucus_bilgisi.get("annotations", [])
    if annotations and annotations[0].get("class") == "External Position":
        # Klasör ismini güvenli formata çeviriyoruz (Tarih slash karakterlerini alt çizgiye veya sistemine uygun hale getir kanka)
        # Eğer klasör yapın doğrudan '19_57_23' gibi ise ona göre filtrele
        clean_folder_name = ucus_adi.split("/")[-1] if "/" in ucus_adi else ucus_adi
        gps_attack_folders.append(clean_folder_name)

print(f"📂 Toplam {len(gps_attack_folders)} adet External Position (GPS Spoofing) uçuşu tespit edildi.")
print("⏳ Analiz için ilk 10 saldırı uçuşu birleştiriliyor ve istatistiki ölçüm yapılıyor...\n")

all_attack_dfs = []
loaded_count = 0

for folder in os.listdir(processed_data_dir):
    if loaded_count >= 10: # İlk 10 tanesi istatistik çıkarmak için fazlasıyla yeterli
        break
    if folder in gps_attack_folders:
        folder_path = os.path.join(processed_data_dir, folder)
        files = os.listdir(folder_path)
        
        imu_file = [f for f in files if f.endswith("sensor_combined_0.csv")]
        air_file = [f for f in files if f.endswith("vehicle_air_data_0.csv")]
        att_file = [f for f in files if f.endswith("vehicle_attitude_0.csv")]
        act_file = [f for f in files if f.endswith("actuator_outputs_0.csv")]
        loc_file = [f for f in files if f.endswith("vehicle_local_position_0.csv")]
        
        if imu_file and air_file and att_file and act_file and loc_file:
            try:
                df_imu = pd.read_csv(os.path.join(folder_path, imu_file[0])).sort_values('timestamp')
                df_air = pd.read_csv(os.path.join(folder_path, air_file[0])).sort_values('timestamp')
                df_att = pd.read_csv(os.path.join(folder_path, att_file[0])).sort_values('timestamp')
                df_act = pd.read_csv(os.path.join(folder_path, act_file[0])).sort_values('timestamp')
                df_loc = pd.read_csv(os.path.join(folder_path, loc_file[0])).sort_values('timestamp')
                
                df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
                df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
                
                if all(col in df_m.columns for col in features_to_train):
                    all_attack_dfs.append(df_m[features_to_train])
                    loaded_count += 1
                    print(f" 🔴 Saldırı Verisi Bağlandı [{loaded_count}/10] -> {folder}")
            except Exception:
                continue

# ==========================================================================================
# 3. İSTATİSTİKSEL SINIR ÖLÇÜMLERİ
# ==========================================================================================
if len(all_attack_dfs) > 0:
    df_attack_master = pd.concat(all_attack_dfs, ignore_index=True).fillna(0)
    
    # Standartlaştırma
    scaler = StandardScaler()
    scaled_attack = scaler.fit_transform(df_attack_master)
    attack_tensor = torch.tensor(scaled_attack, dtype=torch.float32)
    
    # Sabit modelimizden geçirip hataları topluyoruz (Inference)
    with torch.no_grad():
        predictions = model(attack_tensor)
        attack_losses = torch.mean((attack_tensor - predictions) ** 2, dim=1).numpy()
        
    # Matematiksel Dağılım Hesaplama
    mean_attack_loss = np.mean(attack_losses)
    std_attack_loss = np.shadow_std = np.std(attack_losses)
    max_attack_loss = np.max(attack_losses)
    min_attack_loss = np.min(attack_losses)
    
    print("\n==========================================================================================")
    print("📊 EXTERNAL POSITION (GPS SPOOFING) MATEMATİKSEL ANOMALİ RAPORU 📊")
    print("==========================================================================================")
    print(f"🔹 Saldırı Anındaki Ortalama Sapma (Mean Loss): {mean_attack_loss:.6f}")
    print(f"🔹 Saldırı Verilerinin Standart Sapması (Std Dev): {std_attack_loss:.6f}")
    print(f"💥 Saldırının Ulaştığı Tepe Noktası (Max Peak Loss): {max_attack_loss:.6f}")
    print(f"📉 Saldırı Altındaki Taban Değeri (Min Loss): {min_attack_loss:.6f}")
    
    # 3-Sigma Kurallı Akademik Sınır Önerisi
    suggested_threshold = mean_attack_loss - (1.5 * std_attack_loss) # Saldırının başladığı güvenli alt sınır
    print(f"\n📍 Rapora Yazılacak İstatistiki Eşik Değeri (Threshold) Önerisi: {suggested_threshold:.6f}")
    print("==========================================================================================")
else:
    print("❌ Hata: Klasör eşleşmesi sağlanamadı veya işlenecek saldırı verisi bulunamadı.")

📂 Toplam 194 adet External Position (GPS Spoofing) uçuşu tespit edildi.
⏳ Analiz için ilk 10 saldırı uçuşu birleştiriliyor ve istatistiki ölçüm yapılıyor...

 🔴 Saldırı Verisi Bağlandı [1/10] -> 07_58_08

📊 EXTERNAL POSITION (GPS SPOOFING) MATEMATİKSEL ANOMALİ RAPORU 📊
🔹 Saldırı Anındaki Ortalama Sapma (Mean Loss): 0.322265
🔹 Saldırı Verilerinin Standart Sapması (Std Dev): 0.574956
💥 Saldırının Ulaştığı Tepe Noktası (Max Peak Loss): 9.303355
📉 Saldırı Altındaki Taban Değeri (Min Loss): 0.020387

📍 Rapora Yazılacak İstatistiki Eşik Değeri (Threshold) Önerisi: -0.540169


In [8]:
import os

ana_kaynak_dir = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\ulg_files"

print("📂 'ulg_files' İÇİNDEKİ GERÇEK KLASÖR İSİMLERİNDEN ÖRNEKLER:\n")
if os.path.exists(ana_kaynak_dir):
    gercek_klasorler = os.listdir(ana_kaynak_dir)
    print(gercek_klasorler[:5])
else:
    print("❌ Hata: Belirtilen 'ana_kaynak_dir' yolu bilgisayarda bulunamadı!")

📂 'ulg_files' İÇİNDEKİ GERÇEK KLASÖR İSİMLERİNDEN ÖRNEKLER:

['00_02_49.ulg', '00_02_49_actuator_controls_0_0.csv', '00_02_49_actuator_outputs_0.csv', '00_02_49_battery_status_0.csv', '00_02_49_battery_status_1.csv']


In [11]:
import json

mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"

with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("🚨 JSON İÇİNDEKİ GERÇEK SALDIRI ANAHTARLARI (İLK 5 ÖRNEK):\n")

sayac = 0
for ucus_adi, ucus_bilgisi in mapping_data.items():
    annotations = ucus_bilgisi.get("annotations", [])
    if annotations and annotations[0].get("class") == "External Position":
        print(f"➔ JSON Anahtarı: '{ucus_adi}'")
        sayac += 1
        if sayac >= 5:
            break

🚨 JSON İÇİNDEKİ GERÇEK SALDIRI ANAHTARLARI (İLK 5 ÖRNEK):

➔ JSON Anahtarı: '2018-05-24/19_57_23'
➔ JSON Anahtarı: '2018-05-26/16_19_20'
➔ JSON Anahtarı: '2018-05-26/16_48_10'
➔ JSON Anahtarı: '2018-05-26/16_50_01'
➔ JSON Anahtarı: '2018-05-30/18_19_18'


In [1]:
import json

# 1. YOLLARI TANIMLAYALIM
mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"
cikis_txt_yolu = r"C:\Users\feyza\Desktop\uav_project\data\processed\external_position_listesi.txt"

# 2. MAPPING DOSYASINI OKUYALIM
with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("📝 JSON taranıyor ve External Position sınıfına ait uçuşlar ayıklanıyor...\n")

external_position_ucuslari = []

# 3. VERİYİ FİLTRELEYELİM
for ucus_adi, ucus_bilgisi in mapping_data.items():
    annotations = ucus_bilgisi.get("annotations", [])
    if annotations:
        if annotations[0].get("class") == "External Position":
            external_position_ucuslari.append(ucus_adi)

# 4. BULDUĞUMUZ İSİMLERİ TXT DOSYASINA YAZALIM
with open(cikis_txt_yolu, "w", encoding="utf-8") as txt_file:
    for ucus in external_position_ucuslari:
        txt_file.write(f"{ucus}\n")

print("==========================================================================================")
print("🏆 1. ADIM BAŞARIYLA TAMAMLANDI!")
print(f"📊 Toplam {len(external_position_ucuslari)} adet 'External Position' uçuşu TXT'ye yazıldı.")
print(f"📍 TXT Dosya Konumu: {cikis_txt_yolu}")
print("==========================================================================================")

📝 JSON taranıyor ve External Position sınıfına ait uçuşlar ayıklanıyor...

🏆 1. ADIM BAŞARIYLA TAMAMLANDI!
📊 Toplam 194 adet 'External Position' uçuşu TXT'ye yazıldı.
📍 TXT Dosya Konumu: C:\Users\feyza\Desktop\uav_project\data\processed\external_position_listesi.txt


In [4]:
import os
import json

mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"
ana_kaynak_dir = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\ulg_files"

# 1. Klasördeki gerçek alt klasör isimlerini alalım
gercek_klasorler = [f for f in os.listdir(ana_kaynak_dir) if os.path.isdir(os.path.join(ana_kaynak_dir, f))]

print(f"📊 'ulg_files' içinde toplam {len(gercek_klasorler)} adet alt klasör bulundu.")
print("📁 İlk 5 alt klasörün adı çıplak gözle şunlar:")
print(gercek_klasorler[:5])
print("-" * 80)

# 2. Mapping dosyasından 'External Position' olan ilk birkaç örneğe bakalım
with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("🚨 Mapping içindeki ilk 5 External Position anahtarı şunlar:")
sayac = 0
for ucus_adi in mapping_data.keys():
    annotations = mapping_data[ucus_adi].get("annotations", [])
    if annotations and annotations[0].get("class") == "External Position":
        print(f" ➔ JSON'da yazan: '{ucus_adi}'")
        sayac += 1
        if sayac >= 5:
            break

📊 'ulg_files' içinde toplam 263 adet alt klasör bulundu.
📁 İlk 5 alt klasörün adı çıplak gözle şunlar:
['2018-05-24', '2018-05-26', '2018-05-30', '2018-05-31', '2018-06-02']
--------------------------------------------------------------------------------
🚨 Mapping içindeki ilk 5 External Position anahtarı şunlar:
 ➔ JSON'da yazan: '2018-05-24/19_57_23'
 ➔ JSON'da yazan: '2018-05-26/16_19_20'
 ➔ JSON'da yazan: '2018-05-26/16_48_10'
 ➔ JSON'da yazan: '2018-05-26/16_50_01'
 ➔ JSON'da yazan: '2018-05-30/18_19_18'


In [5]:
import os
import json
import shutil

# ==========================================================================================
# YOLLARI TANIMLIYORUZ
# ==========================================================================================
mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"
ana_kaynak_dir = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\ulg_files"
hedef_saldiri_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_categories\External_Position"

os.makedirs(hedef_saldiri_dir, exist_ok=True)

# 1. MAPPING DOSYASINI OKUYALIM
with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("🚀 Gün klasörleri taranıyor ve saat kodlu siber saldırı CSV'leri toplanıyor...\n")

kopyalanan_dosya_sayisi = 0
kurtarilan_ucus_sayisi = 0

# 2. HEDEF TARAMA MOTORU
for ucus_yolu, ucus_bilgisi in mapping_data.items():
    annotations = ucus_bilgisi.get("annotations", [])
    if annotations and annotations[0].get("class") == "External Position":
        
        # Eğer yol '2018-05-24/19_57_23' şeklindeyse parçalıyoruz
        if "/" in ucus_yolu:
            parcalar = ucus_yolu.split("/")
            gun_klasoru = parcalar[0]   # Örn: 2018-05-24
            saat_kodu = parcalar[1]     # Örn: 19_57_23
            # Eğer uzantı kalmışsa temizle
            saat_kodu = saat_kodu.replace(".ulg", "")
            
            # Gerçek fiziksel gün klasörünün yolu
            # Örn: C:\...\ulg_files\2018-05-24
            gercek_gun_dir = os.path.join(ana_kaynak_dir, gun_klasoru)
            
            # Eğer o gün klasörü bilgisayarda fiziksel olarak varsa içine sızıyoruz
            if os.path.exists(gercek_gun_dir) and os.path.isdir(gercek_gun_dir):
                kurtarilan_ucus_sayisi += 1
                
                # Gün klasörünün içindeki tüm dosyaları tara
                for dosya in os.listdir(gercek_gun_dir):
                    # Kural: Dosya adı o saat koduyla mı başlıyor ve CSV mi?
                    if dosya.startswith(saat_kodu) and dosya.endswith(".csv"):
                        kaynak_dosya_yolu = os.path.join(gercek_gun_dir, dosya)
                        
                        # Dosya adının çakışmaması için gün ve saat bilgisini isme mühürlüyoruz
                        yeni_dosya_adi = f"{gun_klasoru}_{dosya}"
                        hedef_dosya_yolu = os.path.join(hedef_saldiri_dir, yeni_dosya_adi)
                        
                        if not os.path.exists(hedef_dosya_yolu):
                            shutil.copy2(kaynak_dosya_yolu, hedef_dosya_yolu)
                            kopyalanan_dosya_sayisi += 1
                            
                if kurtarilan_ucus_sayisi % 10 == 0:
                    print(f" 📦 {kurtarilan_ucus_sayisi} farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...")

print("\n==========================================================================================")
print("🏆 SİBER SALDIRI VERİ AMBARI BAŞARIYLA OLUŞTURULDU!")
print(f"📊 Toplam Başarıyla Erişilen Saldırı Uçuşu: {kurtarilan_ucus_sayisi}")
print(f"📁 Masaüstüne Kopyalanan Toplam Sensör CSV Tablosu: {kopyalanan_dosya_sayisi}")
print(f"📍 Dosyaların Durduğu Havuz: {hedef_saldiri_dir}")
print("==========================================================================================")

🚀 Gün klasörleri taranıyor ve saat kodlu siber saldırı CSV'leri toplanıyor...

 📦 10 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 20 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 30 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 40 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 50 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 60 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 70 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 80 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 90 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 100 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 110 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 120 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıldı...
 📦 130 farklı uçağın saldırı CSV'leri başarıyla masaüstüne aktarıl

In [6]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# ==========================================================================================
# 1. YOLLARI VE MODELİ HAZIRLAMA
# ==========================================================================================
# Artık doğrudan senin o jilet gibi dolan YENİ klasörünü hedef alıyoruz!
attack_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_categories\External_Position"
model_path = r"C:\Users\feyza\Desktop\uav_project\models\uav_autoencoder_v1.pth"

features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

class UAVFinalAutoencoder(nn.Module):
    def __init__(self, dim):
        super(UAVFinalAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Dondurulmuş Modeli Yüklüyoruz (Sadece Çıkarım/Inference Yapacak)
model = UAVFinalAutoencoder(dim=30)
model.load_state_dict(torch.load(model_path))
model.eval()

print("🔍 Yeni siber saldırı ambarından örnek uçuş verileri yükleniyor ve birleştiriliyor...")

# ==========================================================================================
# 2. AYNI TEMİZ VERİDEKİ GİBİ SENSÖR VERİLERİNİ HİZALAYIP GRUPLANDIRMA
# ==========================================================================================
all_files = os.listdir(attack_data_dir)

# İsme göre gruplayıp ilk birkaç uçuşu analiz havuzuna alalım (İstatistiki örneklem)
unique_flights = sorted(list(set([f.split("_sensor_combined")[0] for f in all_files if "_sensor_combined" in f])))[:15]

all_dfs = []
for flight_prefix in unique_flights:
    try:
        # İlgili uçuşun sensör parçalarını çekiyoruz
        f_imu = os.path.join(attack_data_dir, f"{flight_prefix}_sensor_combined_0.csv")
        f_air = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_air_data_0.csv")
        f_att = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_attitude_0.csv")
        f_act = os.path.join(attack_data_dir, f"{flight_prefix}_actuator_outputs_0.csv")
        f_loc = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_local_position_0.csv")
        
        if os.path.exists(f_imu) and os.path.exists(f_air) and os.path.exists(f_att) and os.path.exists(f_act) and os.path.exists(f_loc):
            df_imu = pd.read_csv(f_imu).sort_values('timestamp')
            df_air = pd.read_csv(f_air).sort_values('timestamp')
            df_att = pd.read_csv(f_att).sort_values('timestamp')
            df_act = pd.read_csv(f_act).sort_values('timestamp')
            df_loc = pd.read_csv(f_loc).sort_values('timestamp')
            
            df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
            
            if all(col in df_m.columns for col in features_to_train):
                all_dfs.append(df_m[features_to_train])
    except Exception:
        continue

# ==========================================================================================
# 3. MATEMATİKSEL ANALİZ VE RAPORLAMA
# ==========================================================================================
if len(all_dfs) > 0:
    df_attack_master = pd.concat(all_dfs, ignore_index=True).fillna(0)
    print(f"📊 Toplam {df_attack_master.shape[0]} satırlık GPS Spoofing verisi teraziye alınıyor...")
    
    scaler = StandardScaler()
    scaled_attack = scaler.fit_transform(df_attack_master)
    attack_tensor = torch.tensor(scaled_attack, dtype=torch.float32)
    
    with torch.no_grad():
        predictions = model(attack_tensor)
        attack_losses = torch.mean((attack_tensor - predictions) ** 2, dim=1).numpy()
    
    # Bilimsel Değerleri Çıkarıyoruz
    mean_loss = np.mean(attack_losses)
    std_loss = np.std(attack_losses)
    max_loss = np.max(attack_losses)
    
    print("\n==========================================================================================")
    print("📊 KANITLI GPS SPOOFING (EXTERNAL POSITION) ANOMALİ SINIR RAPORU 📊")
    print("==========================================================================================")
    print(f"🔹 Saldırı Altındaki Gerçek Ortalama Sapma (Mean Loss): {mean_loss:.6f}")
    print(f"🔹 Saldırı Verilerinin Gerçek Standart Sapması (Std Dev): {std_loss:.6f}")
    print(f"💥 Saldırı Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): {max_loss:.6f}")
    
    # Bilimsel Sınır Belirleme Kuralları
    print("\n📍 RAPORA VE JÜRİYE SUNULACAK AKADEMİK EŞİK DEĞERLERİ:")
    print(f"   ➔ Katı/Agresif Sınır (Mean - 1*Std) : {max(0.05, mean_loss - std_loss):.6f}")
    print(f"   ➔ Dengeli/Önerilen Sınır (Mean - 0.5*Std) : {mean_loss - (0.5 * std_loss):.6f}")
    print("==========================================================================================")
else:
    print("❌ Hata: Veriler gruplanırken bir uyuşmazlık oldu.")

🔍 Yeni siber saldırı ambarından örnek uçuş verileri yükleniyor ve birleştiriliyor...
📊 Toplam 40887 satırlık GPS Spoofing verisi teraziye alınıyor...

📊 KANITLI GPS SPOOFING (EXTERNAL POSITION) ANOMALİ SINIR RAPORU 📊
🔹 Saldırı Altındaki Gerçek Ortalama Sapma (Mean Loss): 0.331763
🔹 Saldırı Verilerinin Gerçek Standart Sapması (Std Dev): 0.291344
💥 Saldırı Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): 2.508717

📍 RAPORA VE JÜRİYE SUNULACAK AKADEMİK EŞİK DEĞERLERİ:
   ➔ Katı/Agresif Sınır (Mean - 1*Std) : 0.050000
   ➔ Dengeli/Önerilen Sınır (Mean - 0.5*Std) : 0.186091


In [7]:
import os
import json
import shutil

# ==========================================================================================
# YOLLARI TANIMLIYORUZ (SİZİN SİSTEME TAM ENTEGRE)
# ==========================================================================================
mapping_json_yolu = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\mapping.json"
ana_kaynak_dir = r"C:\Users\feyza\UAV_Anomaly_Project\data\raw\uav_sead\ulg_files"
hedef_ana_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_categories"

# İşleyeceğimiz diğer 3 ana anomali kategorisi
hedef_kategoriler = ["Altitude", "Mechanical", "Global Position"]

# 1. MAPPING DOSYASINI OKUYALIM
with open(mapping_json_yolu, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("🚀 Kalan siber saldırı ve arıza kategorileri için ayıklama işlemi başladı...\n")

# Her kategori için sayaçları tutalım
kategori_istatistikleri = {kat: {"klasor": 0, "dosya": 0} for kat in hedef_kategoriler}

# 2. HEDEF TARAMA VE KOPYALAMA MOTORU
for ucus_yolu, ucus_bilgisi in mapping_data.items():
    annotations = ucus_bilgisi.get("annotations", [])
    if annotations:
        current_class = annotations[0].get("class", "Normal")
        
        # Eğer mevcut uçuş aradığımız 3 kategoriden birine aitse
        if current_class in hedef_kategoriler:
            # Klasör adını Windows uyumlu hale getirmek için boşluğu alt çizgi yapalım (Örn: Global_Position)
            safe_kat_adi = current_class.replace(" ", "_")
            kategori_hedef_dir = os.path.join(hedef_ana_dir, safe_kat_adi)
            os.makedirs(kategori_hedef_dir, exist_ok=True)
            
            # Yol '2018-05-24/19_57_23' şeklindeyse gün ve saat olarak parçalıyoruz
            if "/" in ucus_yolu:
                parcalar = ucus_yolu.split("/")
                gun_klasoru = parcalar[0]
                saat_kodu = parcalar[1].replace(".ulg", "")
                
                # Orijinal bilgisayardaki gün klasörünün yolu
                gercek_gun_dir = os.path.join(ana_kaynak_dir, gun_klasoru)
                
                # Eğer o gün klasörü varsa içine sızıyoruz
                if os.path.exists(gercek_gun_dir) and os.path.isdir(gercek_gun_dir):
                    klasor_sayildi = False
                    
                    for dosya in os.listdir(gercek_gun_dir):
                        # Kural: Dosya adı o saat koduyla mı başlıyor ve CSV mi?
                        if dosya.startswith(saat_kodu) and dosya.endswith(".csv"):
                            if not klasor_sayildi:
                                kategori_istatistikleri[current_class]["klasor"] += 1
                                klasor_sayildi = True
                                
                            kaynak_dosya_yolu = os.path.join(gercek_gun_dir, dosya)
                            
                            # İsim çakışmasını önlemek için günü isme mühürlüyoruz
                            yeni_dosya_adi = f"{gun_klasoru}_{dosya}"
                            hedef_dosya_yolu = os.path.join(kategori_hedef_dir, yeni_dosya_adi)
                            
                            if not os.path.exists(hedef_dosya_yolu):
                                shutil.copy2(kaynak_dosya_yolu, hedef_dosya_yolu)
                                kategori_istatistikleri[current_class]["dosya"] += 1

# ==========================================================================================
# 3. ÖZET ÖLÇÜM RAPORU
# ==========================================================================================
print("\n==========================================================================================")
print("🏆 TÜM SİBER TEHDİT VE ARIZA VERİ AMBARI FİZİKSEL OLARAK KURULDU!")
print("==========================================================================================")
for kat in hedef_kategoriler:
    safe_name = kat.replace(" ", "_")
    print(f"🚨 Kategori: {kat}")
    print(f"   ↳ Başarıyla Erişilen Klasör: {kategori_istatistikleri[kat]['klasor']}")
    print(f"   ↳ Masaüstüne Çekilen CSV Tablosu: {kategori_istatistikleri[kat]['dosya']}")
    print(f"   ↳ Depolandığı Klasör: ...\\attack_categories\\{safe_name}")
    print("-" * 90)

🚀 Kalan siber saldırı ve arıza kategorileri için ayıklama işlemi başladı...


🏆 TÜM SİBER TEHDİT VE ARIZA VERİ AMBARI FİZİKSEL OLARAK KURULDU!
🚨 Kategori: Altitude
   ↳ Başarıyla Erişilen Klasör: 65
   ↳ Masaüstüne Çekilen CSV Tablosu: 2301
   ↳ Depolandığı Klasör: ...\attack_categories\Altitude
------------------------------------------------------------------------------------------
🚨 Kategori: Mechanical
   ↳ Başarıyla Erişilen Klasör: 43
   ↳ Masaüstüne Çekilen CSV Tablosu: 1628
   ↳ Depolandığı Klasör: ...\attack_categories\Mechanical
------------------------------------------------------------------------------------------
🚨 Kategori: Global Position
   ↳ Başarıyla Erişilen Klasör: 41
   ↳ Masaüstüne Çekilen CSV Tablosu: 1390
   ↳ Depolandığı Klasör: ...\attack_categories\Global_Position
------------------------------------------------------------------------------------------


In [8]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# ==========================================================================================
# 1. SADECE GLOBAL_POSITION YOLLARINI VE MODELİ TANIMLIYORUZ
# ==========================================================================================
attack_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_categories\Global_Position"
model_path = r"C:\Users\feyza\Desktop\uav_project\models\uav_autoencoder_v1.pth"

features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

class UAVFinalAutoencoder(nn.Module):
    def __init__(self, dim):
        super(UAVFinalAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = UAVFinalAutoencoder(dim=30)
model.load_state_dict(torch.load(model_path))
model.eval()

print("🔍 Global_Position (GPS Jamming) siber saldırı verileri yükleniyor...")

# ==========================================================================================
# 2. SENSÖR VERİLERİNİ HİZALAYIP ÖRNEKLEM OLUŞTURMA
# ==========================================================================================
all_files = os.listdir(attack_data_dir)
unique_flights = sorted(list(set([f.split("_sensor_combined")[0] for f in all_files if "_sensor_combined" in f])))[:15]

all_dfs = []
for flight_prefix in unique_flights:
    try:
        f_imu = os.path.join(attack_data_dir, f"{flight_prefix}_sensor_combined_0.csv")
        f_air = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_air_data_0.csv")
        f_att = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_attitude_0.csv")
        f_act = os.path.join(attack_data_dir, f"{flight_prefix}_actuator_outputs_0.csv")
        f_loc = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_local_position_0.csv")
        
        if os.path.exists(f_imu) and os.path.exists(f_air) and os.path.exists(f_att) and os.path.exists(f_act) and os.path.exists(f_loc):
            df_imu = pd.read_csv(f_imu).sort_values('timestamp')
            df_air = pd.read_csv(f_air).sort_values('timestamp')
            df_att = pd.read_csv(f_att).sort_values('timestamp')
            df_act = pd.read_csv(f_act).sort_values('timestamp')
            df_loc = pd.read_csv(f_loc).sort_values('timestamp')
            
            df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
            
            if all(col in df_m.columns for col in features_to_train):
                all_dfs.append(df_m[features_to_train])
    except Exception:
        continue

# ==========================================================================================
# 3. İSTATİSTİKSEL HESAPLAMA
# ==========================================================================================
if len(all_dfs) > 0:
    df_attack_master = pd.concat(all_dfs, ignore_index=True).fillna(0)
    print(f"📊 Toplam {df_attack_master.shape[0]} satırlık GPS Jamming verisi analize alınıyor...")
    
    scaler = StandardScaler()
    scaled_attack = scaler.fit_transform(df_attack_master)
    attack_tensor = torch.tensor(scaled_attack, dtype=torch.float32)
    
    with torch.no_grad():
        predictions = model(attack_tensor)
        attack_losses = torch.mean((attack_tensor - predictions) ** 2, dim=1).numpy()
    
    mean_loss = np.mean(attack_losses)
    std_loss = np.std(attack_losses)
    max_loss = np.max(attack_losses)
    
    print("\n==========================================================================================")
    print("📊 KANITLI GLOBAL_POSITION (GPS JAMMING) ANOMALİ SINIR RAPORU 📊")
    print("==========================================================================================")
    print(f"🔹 Saldırı Altındaki Gerçek Ortalama Sapma (Mean Loss): {mean_loss:.6f}")
    print(f"🔹 Saldırı Verilerinin Gerçek Standart Sapması (Std Dev): {std_loss:.6f}")
    print(f"💥 Saldırı Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): {max_loss:.6f}")
    
    print("\n📍 BU KATEGORİ İÇİN ÖNERİLEN AKADEMİK EŞİK DEĞERİ:")
    print(f"   ➔ Dengeli Sınır (Mean - 0.5*Std) : {mean_loss - (0.5 * std_loss):.6f}")
    print("==========================================================================================")
else:
    print("❌ Hata: Global_Position klasöründeki veriler yüklenirken bir sorun oluştu.")

🔍 Global_Position (GPS Jamming) siber saldırı verileri yükleniyor...
📊 Toplam 91191 satırlık GPS Jamming verisi analize alınıyor...

📊 KANITLI GLOBAL_POSITION (GPS JAMMING) ANOMALİ SINIR RAPORU 📊
🔹 Saldırı Altındaki Gerçek Ortalama Sapma (Mean Loss): 0.383529
🔹 Saldırı Verilerinin Gerçek Standart Sapması (Std Dev): 0.792052
💥 Saldırı Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): 8.042128

📍 BU KATEGORİ İÇİN ÖNERİLEN AKADEMİK EŞİK DEĞERİ:
   ➔ Dengeli Sınır (Mean - 0.5*Std) : -0.012497


In [9]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# ==========================================================================================
# 1. SADECE ALTITUDE YOLLARINI VE MODELİ TANIMLIYORUZ
# ==========================================================================================
attack_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_categories\Altitude"
model_path = r"C:\Users\feyza\Desktop\uav_project\models\uav_autoencoder_v1.pth"

features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

class UAVFinalAutoencoder(nn.Module):
    def __init__(self, dim):
        super(UAVFinalAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = UAVFinalAutoencoder(dim=30)
model.load_state_dict(torch.load(model_path))
model.eval()

print("🔍 Altitude (İrtifa Manipülasyonu) siber veri paketleri yükleniyor...")

# ==========================================================================================
# 2. SENSÖR VERİLERİNİ HİZALAYIP ÖRNEKLEM OLUŞTURMA
# ==========================================================================================
all_files = os.listdir(attack_data_dir)
unique_flights = sorted(list(set([f.split("_sensor_combined")[0] for f in all_files if "_sensor_combined" in f])))[:15]

all_dfs = []
for flight_prefix in unique_flights:
    try:
        f_imu = os.path.join(attack_data_dir, f"{flight_prefix}_sensor_combined_0.csv")
        f_air = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_air_data_0.csv")
        f_att = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_attitude_0.csv")
        f_act = os.path.join(attack_data_dir, f"{flight_prefix}_actuator_outputs_0.csv")
        f_loc = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_local_position_0.csv")
        
        if os.path.exists(f_imu) and os.path.exists(f_air) and os.path.exists(f_att) and os.path.exists(f_act) and os.path.exists(f_loc):
            df_imu = pd.read_csv(f_imu).sort_values('timestamp')
            df_air = pd.read_csv(f_air).sort_values('timestamp')
            df_att = pd.read_csv(f_att).sort_values('timestamp')
            df_act = pd.read_csv(f_act).sort_values('timestamp')
            df_loc = pd.read_csv(f_loc).sort_values('timestamp')
            
            df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
            
            if all(col in df_m.columns for col in features_to_train):
                all_dfs.append(df_m[features_to_train])
    except Exception:
        continue

# ==========================================================================================
# 3. İSTATİSTİKSEL HESAPLAMA
# ==========================================================================================
if len(all_dfs) > 0:
    df_attack_master = pd.concat(all_dfs, ignore_index=True).fillna(0)
    print(f"📊 Toplam {df_attack_master.shape[0]} satırlık İrtifa Saldırı verisi teraziye alınıyor...")
    
    scaler = StandardScaler()
    scaled_attack = scaler.fit_transform(df_attack_master)
    attack_tensor = torch.tensor(scaled_attack, dtype=torch.float32)
    
    with torch.no_grad():
        predictions = model(attack_tensor)
        attack_losses = torch.mean((attack_tensor - predictions) ** 2, dim=1).numpy()
    
    mean_loss = np.mean(attack_losses)
    std_loss = np.std(attack_losses)
    max_loss = np.max(attack_losses)
    
    print("\n==========================================================================================")
    print("📊 KANITLI ALTITUDE (İRTİFA MANİPÜLASYONU) ANOMALİ SINIR RAPORU 📊")
    print("==========================================================================================")
    print(f"🔹 Saldırı Altındaki Gerçek Ortalama Sapma (Mean Loss): {mean_loss:.6f}")
    print(f"🔹 Saldırı Verilerinin Gerçek Standart Sapması (Std Dev): {std_loss:.6f}")
    print(f"💥 Saldırı Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): {max_loss:.6f}")
    
    print("\n📍 BU KATEGORİ İÇİN ÖNERİLEN AKADEMİK EŞİK DEĞERİ:")
    print(f"   ➔ Dengeli Sınır (Mean - 0.5*Std) : {mean_loss - (0.5 * std_loss):.6f}")
    print("==========================================================================================")
else:
    print("❌ Hata: Altitude klasöründeki veriler işlenirken bir sorun oluştu.")

🔍 Altitude (İrtifa Manipülasyonu) siber veri paketleri yükleniyor...
📊 Toplam 372234 satırlık İrtifa Saldırı verisi teraziye alınıyor...

📊 KANITLI ALTITUDE (İRTİFA MANİPÜLASYONU) ANOMALİ SINIR RAPORU 📊
🔹 Saldırı Altındaki Gerçek Ortalama Sapma (Mean Loss): 0.406480
🔹 Saldırı Verilerinin Gerçek Standart Sapması (Std Dev): 0.749366
💥 Saldırı Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): 9.917249

📍 BU KATEGORİ İÇİN ÖNERİLEN AKADEMİK EŞİK DEĞERİ:
   ➔ Dengeli Sınır (Mean - 0.5*Std) : 0.031797


In [10]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# ==========================================================================================
# 1. SADECE MECHANICAL YOLLARINI VE MODELİ TANIMLIYORUZ
# ==========================================================================================
attack_data_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_categories\Mechanical"
model_path = r"C:\Users\feyza\Desktop\uav_project\models\uav_autoencoder_v1.pth"

features_to_train = [
    'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]',
    'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]',
    'baro_alt_meter', 'baro_pressure_pa', 'baro_temp_celcius',
    'q[0]', 'q[1]', 'q[2]', 'q[3]',
    'output[0]', 'output[1]', 'output[2]', 'output[3]', 'output[4]', 'output[5]', 'output[6]', 'output[7]',
    'x', 'y', 'z', 'vx', 'vy', 'vz', 'ax', 'ay', 'az'
]

class UAVFinalAutoencoder(nn.Module):
    def __init__(self, dim):
        super(UAVFinalAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, dim)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = UAVFinalAutoencoder(dim=30)
model.load_state_dict(torch.load(model_path))
model.eval()

print("🔍 Mechanical (Donanım/Motor Arızaları) veri paketleri teraziye alınıyor...")

# ==========================================================================================
# 2. SENSÖR VERİLERİNİ HİZALAYIP ÖRNEKLEM OLUŞTURMA
# ==========================================================================================
all_files = os.listdir(attack_data_dir)
unique_flights = sorted(list(set([f.split("_sensor_combined")[0] for f in all_files if "_sensor_combined" in f])))[:15]

all_dfs = []
for flight_prefix in unique_flights:
    try:
        f_imu = os.path.join(attack_data_dir, f"{flight_prefix}_sensor_combined_0.csv")
        f_air = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_air_data_0.csv")
        f_att = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_attitude_0.csv")
        f_act = os.path.join(attack_data_dir, f"{flight_prefix}_actuator_outputs_0.csv")
        f_loc = os.path.join(attack_data_dir, f"{flight_prefix}_vehicle_local_position_0.csv")
        
        if os.path.exists(f_imu) and os.path.exists(f_air) and os.path.exists(f_att) and os.path.exists(f_act) and os.path.exists(f_loc):
            df_imu = pd.read_csv(f_imu).sort_values('timestamp')
            df_air = pd.read_csv(f_air).sort_values('timestamp')
            df_att = pd.read_csv(f_att).sort_values('timestamp')
            df_act = pd.read_csv(f_act).sort_values('timestamp')
            df_loc = pd.read_csv(f_loc).sort_values('timestamp')
            
            df_m = pd.merge_asof(df_imu, df_air, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_att, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_act, on='timestamp', direction='nearest')
            df_m = pd.merge_asof(df_m, df_loc, on='timestamp', direction='nearest')
            
            if all(col in df_m.columns for col in features_to_train):
                all_dfs.append(df_m[features_to_train])
    except Exception:
        continue

# ==========================================================================================
# 3. İSTATİSTİKSEL HESAPLAMA
# ==========================================================================================
if len(all_dfs) > 0:
    df_attack_master = pd.concat(all_dfs, ignore_index=True).fillna(0)
    print(f"📊 Toplam {df_attack_master.shape[0]} satırlık Mekanik Arıza verisi işleniyor...")
    
    scaler = StandardScaler()
    scaled_attack = scaler.fit_transform(df_attack_master)
    attack_tensor = torch.tensor(scaled_attack, dtype=torch.float32)
    
    with torch.no_grad():
        predictions = model(attack_tensor)
        attack_losses = torch.mean((attack_tensor - predictions) ** 2, dim=1).numpy()
    
    mean_loss = np.mean(attack_losses)
    std_loss = np.std(attack_losses)
    max_loss = np.max(attack_losses)
    
    print("\n==========================================================================================")
    print("📊 KANITLI MECHANICAL (DONANIM ARIZASI) ANOMALİ SINIR RAPORU 📊")
    print("==========================================================================================")
    print(f"🔹 Arıza Altındaki Gerçek Ortalama Sapma (Mean Loss): {mean_loss:.6f}")
    print(f"🔹 Arıza Verilerinin Gerçek Standart Sapması (Std Dev): {std_loss:.6f}")
    print(f"💥 Arıza Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): {max_loss:.6f}")
    
    print("\n📍 BU KATEGORİ İÇİN ÖNERİLEN AKADEMİK EŞİK DEĞERİ:")
    print(f"   ➔ Dengeli Sınır (Mean - 0.5*Std) : {mean_loss - (0.5 * std_loss):.6f}")
    print("==========================================================================================")
else:
    print("❌ Hata: Mechanical klasöründeki veriler işlenirken bir sorun oluştu.")

🔍 Mechanical (Donanım/Motor Arızaları) veri paketleri teraziye alınıyor...
📊 Toplam 380567 satırlık Mekanik Arıza verisi işleniyor...

📊 KANITLI MECHANICAL (DONANIM ARIZASI) ANOMALİ SINIR RAPORU 📊
🔹 Arıza Altındaki Gerçek Ortalama Sapma (Mean Loss): 0.347844
🔹 Arıza Verilerinin Gerçek Standart Sapması (Std Dev): 0.388216
💥 Arıza Sırasında Ulaşılan Maksimum Tepe Noktası (Max Loss): 45.337944

📍 BU KATEGORİ İÇİN ÖNERİLEN AKADEMİK EŞİK DEĞERİ:
   ➔ Dengeli Sınır (Mean - 0.5*Std) : 0.153735


In [11]:
import json
import os

# Konfigürasyon dosyasının kaydedileceği nihai kurumsal patika
config_cikis_yolu = r"C:\Users\feyza\Desktop\uav_project\models\anomaly_config.json"

# Sektör standardında meta-veri ve eşik şeması (Enterprise Configuration Schema)
uav_anomaly_framework_config = {
    "project_metadata": {
        "project_name": "AVERTİA - UAV Anomaly Detection Framework",
        "version": "1.0.0",
        "framework": "PyTorch & Pearsons/Standard Distribution Analysis",
        "target_model_weights": "uav_autoencoder_v1.pth",
        "input_dimension": 30,
        "documentation_culture": "Clean Architecture / Siber Vatan Compliance"
    },
    "global_bounds": {
        "normal_flight_baseline_loss": 0.035000,
        "absolute_clipping_lower_bound": 0.050000
    },
    "anomaly_categories": {
        "External_Position": {
            "threat_type": "Cyber Attack",
            "sub_class": "GPS Spoofing / Location Manipulation",
            "description": "Dış kaynaklı sahte GPS sinyalleri ile İHA'nın yatay düzlemdeki rota ve koordinat verilerinin manipüle edilmesi.",
            "metrics": {
                "mean_reconstruction_loss": 0.331763,
                "standard_deviation": 0.291344,
                "maximum_peak_loss": 2.508717
            },
            "security_policy": {
                "calculated_threshold": 0.186091,
                "applied_mitigation_threshold": 0.186091,
                "alarm_severity": "CRITICAL"
            }
        },
        "Global_Position": {
            "threat_type": "Cyber Attack / Electronic Warfare",
            "sub_class": "GPS Jamming / Signal Loss",
            "description": "Elektronik harp veya sinyal kesiciler vasıtasıyla GPS bağlantısının tamamen koparılması. Yüksek varyanslı gürültü içerir.",
            "metrics": {
                "mean_reconstruction_loss": 0.383529,
                "standard_deviation": 0.792052,
                "maximum_peak_loss": 8.042128
            },
            "security_policy": {
                "calculated_threshold": -0.012497,
                "applied_mitigation_threshold": 0.050000,
                "alarm_severity": "HIGH",
                "note": "Matematiksel eşik negatif değere düştüğü için absolute_clipping_lower_bound (0.05) güvenlik politikası olarak atanmıştır."
            }
        },
        "Altitude": {
            "threat_type": "Cyber Attack / Sensor Defect",
            "sub_class": "Barometer Manipulation / Altitude Attack",
            "description": "Barometrik basınç ve irtifa sensörlerine yönelik siber sızma veya ani dikey eksen veri manipülasyonu.",
            "metrics": {
                "mean_reconstruction_loss": 0.406480,
                "standard_deviation": 0.749366,
                "maximum_peak_loss": 9.917249
            },
            "security_policy": {
                "calculated_threshold": 0.031797,
                "applied_mitigation_threshold": 0.031797,
                "alarm_severity": "CRITICAL"
            }
        },
        "Mechanical": {
            "threat_type": "Hardware Failure",
            "sub_class": "Actuator / Motor / Control Surface Malfunction",
            "description": "İHA'nın motor, pervane veya servo kanatçıklarında meydana gelen fiziksel/donanımsal kilitlenmeler ve arızalar.",
            "metrics": {
                "mean_reconstruction_loss": 0.347844,
                "standard_deviation": 0.388216,
                "maximum_peak_loss": 45.337944
            },
            "security_policy": {
                "calculated_threshold": 0.153735,
                "applied_mitigation_threshold": 0.153735,
                "alarm_severity": "EMERGENCY"
            }
        }
    }
}

# JSON dosyasını okunaklı (indent=4) ve Türkçe karakter korumalı olarak yazıyoruz
os.makedirs(os.path.dirname(config_cikis_yolu), exist_ok=True)
with open(config_cikis_yolu, "w", encoding="utf-8") as json_file:
    json.dump(uav_anomaly_framework_config, json_file, indent=4, ensure_ascii=False)

print("\n==========================================================================================")
print("🏆 SEKTÖR STANDARDINDA MERKEZİ SİBER GÜVENLİK JSON YAPILANDIRMASI YAZILDI!")
print(f"📍 Kayıt Adresi: {config_cikis_yolu}")
print("==========================================================================================")


🏆 SEKTÖR STANDARDINDA MERKEZİ SİBER GÜVENLİK JSON YAPILANDIRMASI YAZILDI!
📍 Kayıt Adresi: C:\Users\feyza\Desktop\uav_project\models\anomaly_config.json
